In [ ]:
import os
import glob
from collections import defaultdict
from fontTools.ttLib import TTFont
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

FONT_DIR = "./fonts/ttf"
NUM_WORKERS = 32

# ── 문자 정의 ──
digits          = [chr(c) for c in range(0x30, 0x3A)]         # 0-9
korean_jamo     = [chr(c) for c in range(0x3131, 0x3164)]     # ㄱ-ㅣ
korean_syllables = []
for code in tqdm(range(0xAC00, 0xD7A4), desc="한글 음절 필터링"):
    ch = chr(code)
    try:
        encoded = ch.encode("euc_kr")
        if len(encoded) == 2 and 0xB0 <= encoded[0] <= 0xC8 and 0xA1 <= encoded[1] <= 0xFE:
            korean_syllables.append(ch)
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
latin = [chr(c) for c in range(0x41, 0x5B)] + [chr(c) for c in range(0x61, 0x7B)]

ALL_CHARS = digits + korean_jamo + korean_syllables + latin
ALL_CODEPOINTS = [ord(c) for c in ALL_CHARS]

cat_codepoints = {
    "숫자":     set(ord(c) for c in digits),
    "한글낱자": set(ord(c) for c in korean_jamo),
    "글자마디": set(ord(c) for c in korean_syllables),
    "로마":     set(ord(c) for c in latin),
}

print(f"문자 구성: 숫자 {len(digits)} + 낱자 {len(korean_jamo)} + 음절 {len(korean_syllables)} + 로마 {len(latin)} = {len(ALL_CHARS)}자")


def check_font(args):
    path, codepoints = args
    fname = os.path.splitext(os.path.basename(path))[0]
    result = {"file": fname, "path": path, "error": None,
              "missing_cmap": [], "empty_glyph": []}
    try:
        tt = TTFont(path)
        cmap = tt.getBestCmap()
        if cmap is None:
            result["error"] = "no cmap"
            tt.close()
            return result

        has_glyf = "glyf" in tt
        has_cff  = "CFF " in tt or "CFF2" in tt
        glyf     = tt["glyf"] if has_glyf else None

        for cp in codepoints:
            if cp not in cmap:
                result["missing_cmap"].append(cp)
                continue
            glyph_name = cmap[cp]
            if has_glyf:
                if glyph_name not in glyf:
                    result["empty_glyph"].append(cp)
                else:
                    g = glyf[glyph_name]
                    if g.numberOfContours == 0 and not g.isComposite():
                        result["empty_glyph"].append(cp)
        tt.close()
    except Exception as e:
        result["error"] = str(e)
    return result


# ── 병렬 검증 ──
font_files = sorted(glob.glob(os.path.join(FONT_DIR, "*.ttf")))
print(f"총 폰트 파일: {len(font_files)}개\n")

tasks = [(p, ALL_CODEPOINTS) for p in font_files]
results = []
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(check_font, t): t for t in tasks}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="폰트 검증"):
        results.append(fut.result())

# ── 폰트명별 집계 ──
exclude_names = set()
file_issues = []
for r in results:
    font_name = r["file"].rsplit("-", 1)[0]
    n_bad = len(r["missing_cmap"]) + len(r["empty_glyph"]) + (1 if r["error"] else 0)
    if n_bad > 0:
        exclude_names.add(font_name)
        # 카테고리별 누락 집계
        missing_set = set(r["missing_cmap"]) | set(r["empty_glyph"])
        file_issues.append({
            "file": r["file"],
            "font_name": font_name,
            "error": r["error"],
            "total_bad": len(missing_set) if not r["error"] else -1,
            "bad_digits": len(missing_set & cat_codepoints["숫자"]),
            "bad_jamo": len(missing_set & cat_codepoints["한글낱자"]),
            "bad_syll": len(missing_set & cat_codepoints["글자마디"]),
            "bad_latin": len(missing_set & cat_codepoints["로마"]),
        })

all_font_names = set(r["file"].rsplit("-", 1)[0] for r in results)
usable = sorted(all_font_names - exclude_names)

print(f"\n{'='*60}")
print(f"전체 폰트:      {len(all_font_names)}종 ({len(results)}파일)")
print(f"제외:           {len(exclude_names)}종")
print(f"최종 사용 가능: {len(usable)}종")
print(f"{'='*60}")

if file_issues:
    file_issues.sort(key=lambda x: x["total_bad"], reverse=True)
    print(f"\n제외 폰트 상세 (파일 단위):")
    print(f"  {'파일':<45} {'문제':>5} {'숫자':>4} {'낱자':>4} {'음절':>4} {'로마':>4} {'에러'}")
    for fi in file_issues:
        err_str = fi["error"] or ""
        print(f"  {fi['file']:<45} {fi['total_bad']:>5} {fi['bad_digits']:>4} {fi['bad_jamo']:>4} {fi['bad_syll']:>4} {fi['bad_latin']:>4} {err_str}")

In [ ]:
import os
import glob
from PIL import Image, ImageDraw, ImageFont
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

FONT_DIR = "./fonts/ttf"
OUTPUT_DIR = "./data/9_font_image"
NUM_WORKERS = 32
IMG_SIZE = 224
PADDING = 8  # 좌우상하 여백
MAX_AREA = IMG_SIZE - PADDING * 2  # 글자가 차지할 수 있는 최대 영역
INIT_FONT_SIZE = 200  # bbox 측정용 초기 크기

# ── 문자 정의 ──
digits          = [chr(c) for c in range(0x30, 0x3A)]
korean_jamo     = [chr(c) for c in range(0x3131, 0x3164)]
korean_syllables = []
for code in range(0xAC00, 0xD7A4):
    ch = chr(code)
    try:
        encoded = ch.encode("euc_kr")
        if len(encoded) == 2 and 0xB0 <= encoded[0] <= 0xC8 and 0xA1 <= encoded[1] <= 0xFE:
            korean_syllables.append(ch)
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
latin = [chr(c) for c in range(0x41, 0x5B)] + [chr(c) for c in range(0x61, 0x7B)]
ALL_CHARS = digits + korean_jamo + korean_syllables + latin
print(f"문자 수: {len(ALL_CHARS)}자")

# ── 제외 폰트 ──
EXCLUDE = {
    '116angduk_honesty', '116angmuburi', '116watermelon', 'BusanFont',
    'CAChildrenKkum', 'CAChildrenModu', 'Cafe24PROUP', 'Chilgok_Kaj',
    'DOSPilgi', 'Danjo', 'DaehwaNanum-sound', 'Diphylleia',
    'Gilbeot_Rainbow', 'KCC-Ahnjunggeun', 'KCC-BangJeonghwan', 'KJDcom',
    'NationalLibrary', 'PartialSans', 'PeachMarket', 'gabia-solmee',
    'locus_sangsang', 'tillvictorycomes',
}

# ── 사용 가능 폰트 파일 목록 ──
font_files = sorted(glob.glob(os.path.join(FONT_DIR, "*.ttf")))
usable_files = []
for path in font_files:
    fname = os.path.splitext(os.path.basename(path))[0]
    font_name = fname.rsplit("-", 1)[0]
    if font_name not in EXCLUDE:
        weight = fname.rsplit("-", 1)[1]
        usable_files.append({"path": path, "font_name": font_name, "weight": weight})

print(f"사용 가능 폰트 파일: {len(usable_files)}개")
print(f"총 렌더링 이미지: {len(usable_files) * len(ALL_CHARS):,}장")


def render_font(args):
    font_path, font_name, weight, chars, img_size, padding, max_area, init_fs = args
    out_dir = os.path.join(OUTPUT_DIR, font_name, weight)
    os.makedirs(out_dir, exist_ok=True)

    existing = set(os.listdir(out_dir))
    if len(existing) >= len(chars):
        return {"font_name": font_name, "weight": weight, "rendered": 0, "skipped": len(chars), "errors": 0}

    # 측정용 폰트 로드
    try:
        ref_font = ImageFont.truetype(font_path, init_fs)
    except Exception:
        return {"font_name": font_name, "weight": weight, "rendered": 0, "skipped": 0, "errors": len(chars)}

    # 글자별 최적 font_size 캐시 (같은 폰트 내에서 재사용)
    font_cache = {}
    ref_draw = ImageDraw.Draw(Image.new("L", (1, 1)))

    rendered = 0
    errors = 0
    skipped = 0

    for ch in chars:
        fname = f"{ord(ch):04X}.png"
        if fname in existing:
            skipped += 1
            continue

        try:
            # 1) 초기 크기로 bbox 측정
            bbox = ref_draw.textbbox((0, 0), ch, font=ref_font)
            bw = bbox[2] - bbox[0]
            bh = bbox[3] - bbox[1]

            if bw <= 0 or bh <= 0:
                errors += 1
                continue

            # 2) 최적 font_size 계산
            scale = min(max_area / bw, max_area / bh)
            optimal_fs = max(1, int(init_fs * scale))

            # 3) 최적 크기 폰트 로드 (캐시)
            if optimal_fs not in font_cache:
                font_cache[optimal_fs] = ImageFont.truetype(font_path, optimal_fs)
            font = font_cache[optimal_fs]

            # 4) 렌더링
            img = Image.new("L", (img_size, img_size), 255)
            draw = ImageDraw.Draw(img)
            bbox2 = draw.textbbox((0, 0), ch, font=font)
            tw = bbox2[2] - bbox2[0]
            th = bbox2[3] - bbox2[1]
            x = (img_size - tw) / 2 - bbox2[0]
            y = (img_size - th) / 2 - bbox2[1]
            draw.text((x, y), ch, fill=0, font=font)
            img.save(os.path.join(out_dir, fname))
            rendered += 1
        except Exception:
            errors += 1

    return {"font_name": font_name, "weight": weight, "rendered": rendered, "skipped": skipped, "errors": errors}


# ── 병렬 렌더링 ──
tasks = [(f["path"], f["font_name"], f["weight"], ALL_CHARS,
          IMG_SIZE, PADDING, MAX_AREA, INIT_FONT_SIZE) for f in usable_files]

total_rendered = 0
total_skipped = 0
total_errors = 0
error_fonts = []

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(render_font, t): t for t in tasks}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="렌더링"):
        result = fut.result()
        total_rendered += result["rendered"]
        total_skipped += result["skipped"]
        total_errors += result["errors"]
        if result["errors"] > 0:
            error_fonts.append(f"{result['font_name']}-{result['weight']}: {result['errors']}건")

print(f"\n{'='*50}")
print(f"렌더링 완료: {total_rendered:,}장")
print(f"스킵 (이미 존재): {total_skipped:,}장")
print(f"에러: {total_errors:,}장")
print(f"{'='*50}")

if error_fonts:
    print(f"\n에러 폰트 ({len(error_fonts)}개):")
    for e in error_fonts[:20]:
        print(f"  {e}")

In [ ]:
import os
import glob
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

FONT_IMG_DIR = "./data/9_font_image"
OUTPUT_DIR = "./data/9_font_image_embedding"
MODEL_NAME = "facebook/dinov3-vit7b16-pretrain-lvd1689m"
BATCH_SIZE = 512
DL_WORKERS = 8

# ── 모델 로드 ──
print("모델 로드 중...")
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()
model = torch.compile(model)
print(f"모델 로드 완료 — device: {model.device}, dtype: {model.dtype}")


class FontWeightDataset(Dataset):
    def __init__(self, img_dir, processor):
        self.paths = sorted(glob.glob(os.path.join(img_dir, "*.png")))
        self.processor = processor

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        inputs = self.processor(images=img, return_tensors="pt")
        return inputs["pixel_values"].squeeze(0)


# ── 처리 대상 수집 ──
tasks = []
font_dirs = sorted([d for d in glob.glob(os.path.join(FONT_IMG_DIR, "*")) if os.path.isdir(d)])
for font_dir in font_dirs:
    font_name = os.path.basename(font_dir)
    for weight_dir in sorted([d for d in glob.glob(os.path.join(font_dir, "*")) if os.path.isdir(d)]):
        weight = os.path.basename(weight_dir)
        out_path = os.path.join(OUTPUT_DIR, font_name, f"{weight}.pt")
        if os.path.exists(out_path):
            continue
        tasks.append({"font_name": font_name, "weight": weight, "img_dir": weight_dir, "out_path": out_path})

print(f"폰트 디렉토리: {len(font_dirs)}개")
print(f"처리 대상 weight: {len(tasks)}개 (이미 완료된 것 제외)")

# ── 임베딩 추출 ──
for task in tqdm(tasks, desc="임베딩 추출"):
    dataset = FontWeightDataset(task["img_dir"], processor)
    if len(dataset) == 0:
        continue

    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=DL_WORKERS,
                            pin_memory=True, persistent_workers=True)

    embeddings = []
    with torch.inference_mode():
        for batch in dataloader:
            batch = batch.to(model.device, dtype=torch.bfloat16)
            outputs = model(batch)
            embeddings.append(outputs.pooler_output.cpu().float())

    embeddings = torch.cat(embeddings, dim=0)  # (num_chars, 4096)

    os.makedirs(os.path.dirname(task["out_path"]), exist_ok=True)
    torch.save(embeddings, task["out_path"])

In [1]:
import os
import glob
import torch
from tqdm.auto import tqdm

OUTPUT_DIR = "./data/9_font_image_embedding"

# ── 저장된 .pt 파일 확인 ──
saved = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.pt"), recursive=True))
fonts_done = set(os.path.basename(os.path.dirname(p)) for p in saved)

print(f"저장된 .pt 파일: {len(saved)}개")
print(f"완료된 폰트:     {len(fonts_done)}종")

if saved:
    sample = saved[0]
    t = torch.load(sample, weights_only=True)
    print(f"\n샘플: {os.path.relpath(sample, OUTPUT_DIR)}")
    print(f"  shape: {t.shape}  (기대: (2463, 4096))")
    print(f"  dtype: {t.dtype}")
    print(f"  min: {t.min().item():.4f}, max: {t.max().item():.4f}, mean: {t.mean().item():.4f}")

# ── 폰트별 weight 수 분포 ──
    font_weight_counts = {}
    for p in tqdm(saved, desc="폰트별 weight 집계"):
        font_name = os.path.basename(os.path.dirname(p))
        font_weight_counts[font_name] = font_weight_counts.get(font_name, 0) + 1

    counts = list(font_weight_counts.values())
    print(f"\n폰트별 weight 수: min {min(counts)}, max {max(counts)}, mean {sum(counts)/len(counts):.1f}")

# ── shape 이상 확인 ──
    bad = []
    for p in tqdm(saved, desc="shape 검증"):
        t = torch.load(p, weights_only=True)
        if t.shape != (2463, 4096):
            bad.append((os.path.relpath(p, OUTPUT_DIR), t.shape))
    print(f"\nshape 이상: {len(bad)}개")
    if bad:
        for name, shape in bad[:10]:
            print(f"  {name}: {shape}")

저장된 .pt 파일: 1213개
완료된 폰트:     641종

샘플: 11STREET_Gothic/300.pt
  shape: torch.Size([2463, 4096])  (기대: (2463, 4096))
  dtype: torch.float32
  min: -1.0469, max: 1.2578, mean: 0.0029


폰트별 weight 집계:   0%|          | 0/1213 [00:00<?, ?it/s]


폰트별 weight 수: min 1, max 9, mean 1.9


shape 검증:   0%|          | 0/1213 [00:00<?, ?it/s]


shape 이상: 0개


In [19]:
import os
import glob
import torch
import numpy as np
from collections import defaultdict
from sklearn.cluster import AffinityPropagation
from sklearn.metrics import silhouette_score
from tqdm.auto import tqdm

EMBEDDING_DIR = "./data/9_font_image_embedding"

# ── 1) 글자축 mean → weight축 mean → 폰트당 1벡터 ──
font_dirs = sorted([d for d in glob.glob(os.path.join(EMBEDDING_DIR, "*")) if os.path.isdir(d)])
print(f"폰트 디렉토리: {len(font_dirs)}개")

font_names = []
font_embeddings = []

for font_dir in tqdm(font_dirs, desc="폰트별 mean 임베딩 산출"):
    font_name = os.path.basename(font_dir)
    pt_files = sorted(glob.glob(os.path.join(font_dir, "*.pt")))
    if not pt_files:
        continue

    weight_means = []
    for pt_path in pt_files:
        t = torch.load(pt_path, weights_only=True)  # (2463, 4096)
        weight_means.append(t.mean(dim=0))  # (4096,) 글자축 mean

    font_vec = torch.stack(weight_means).mean(dim=0)  # (4096,) weight축 mean
    font_names.append(font_name)
    font_embeddings.append(font_vec.numpy())

X = np.stack(font_embeddings)  # (N, 4096)
print(f"클러스터링 입력: {X.shape} ({len(font_names)}종 × 4096차원)")

# ── 2) Affinity Propagation 클러스터링 ──
print("\nAffinity Propagation 실행 중...")
ap = AffinityPropagation(preference=-5,random_state=42)
labels = ap.fit_predict(X)

n_clusters = len(set(labels))
exemplar_indices = ap.cluster_centers_indices_

print(f"\n{'='*60}")
print(f"클러스터 수: {n_clusters}")
print(f"Silhouette Score: {silhouette_score(X, labels):.4f}")
print(f"{'='*60}")

# ── 3) 클러스터별 요약 ──
cluster_members = defaultdict(list)
for i, label in enumerate(labels):
    cluster_members[label].append(font_names[i])

print(f"\n클러스터별 폰트 수:")
for cid in sorted(cluster_members.keys()):
    members = cluster_members[cid]
    exemplar_name = font_names[exemplar_indices[cid]]
    print(f"  Cluster {cid:>3d}: {len(members):>3d}개  (대표: {exemplar_name})")

# ── 4) 클러스터 크기 분포 ──
sizes = [len(v) for v in cluster_members.values()]
print(f"\n클러스터 크기 분포:")
print(f"  min: {min(sizes)}, max: {max(sizes)}, mean: {np.mean(sizes):.1f}, median: {np.median(sizes):.1f}")
print(f"  1개짜리 클러스터: {sum(1 for s in sizes if s == 1)}개")

# ── 5) 결과 저장 ──
result = {
    "font_names": font_names,
    "labels": labels.tolist(),
    "exemplar_indices": exemplar_indices.tolist(),
    "n_clusters": n_clusters,
}
save_path = "./data/9_font_clusters.pt"
torch.save(result, save_path)
print(f"\n결과 저장: {save_path}")

폰트 디렉토리: 641개


폰트별 mean 임베딩 산출:   0%|          | 0/641 [00:00<?, ?it/s]

클러스터링 입력: (641, 4096) (641종 × 4096차원)

Affinity Propagation 실행 중...

클러스터 수: 79
Silhouette Score: 0.1408

클러스터별 폰트 수:
  Cluster   0:   4개  (대표: April16th-Safety)
  Cluster   1:   1개  (대표: BM-Euljirooraeorae)
  Cluster   2:   1개  (대표: BM-KIRANGHAERANG)
  Cluster   3:  16개  (대표: BobaesumJindo)
  Cluster   4:   8개  (대표: Chilgok_kyb)
  Cluster   5:  14개  (대표: ChosunGs)
  Cluster   6:   1개  (대표: ChusaLove)
  Cluster   7:   3개  (대표: DNFBitBitv2)
  Cluster   8:   7개  (대표: DNFForgedBlade)
  Cluster   9:   3개  (대표: Dongle)
  Cluster  10:   9개  (대표: Galmuri11)
  Cluster  11:   3개  (대표: Galmuri7)
  Cluster  12:  21개  (대표: GamjaFlower)
  Cluster  13:   1개  (대표: GamtanRoad-Gamtan)
  Cluster  14:   8개  (대표: GangwonEdu)
  Cluster  15:  16개  (대표: GeumcheonGvalleySans)
  Cluster  16:   2개  (대표: Ghanachocolate)
  Cluster  17:   1개  (대표: Giants-inline)
  Cluster  18:  22개  (대표: Godo)
  Cluster  19:  12개  (대표: Goyangllsan)
  Cluster  20:   1개  (대표: Grandiflora-One)
  Cluster  21:   8개  (대표: HBIOSSYS)
  Cl

In [ ]:
import os
import glob
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from collections import defaultdict
from tqdm.auto import tqdm

FONT_DIR = "./fonts/ttf"
CLUSTER_PATH = "./data/9_font_clusters_s.pt"
OUTPUT_DIR = "./data/9_font_cluster_vis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLE_TEXT = "가낡더떴끼릭맺료콤으숲황HaReSg125680"
FONT_SIZE = 40
ROW_HEIGHT = 60
LABEL_WIDTH = 300
TEXT_WIDTH = 900
IMG_WIDTH = LABEL_WIDTH + TEXT_WIDTH
PADDING = 10
BG_COLOR = (255, 255, 255)
FG_COLOR = (0, 0, 0)
EXEMPLAR_BG = (230, 245, 255)

# ── 1) 클러스터 결과 로드 ──
result = torch.load(CLUSTER_PATH, weights_only=False)
font_names = result["font_names"]
labels = result["labels"]
exemplar_indices = result["exemplar_indices"]
n_clusters = result["n_clusters"]

cluster_members = defaultdict(list)
for i, label in enumerate(labels):
    cluster_members[label].append(font_names[i])

print(f"클러스터 수: {n_clusters}, 폰트 수: {len(font_names)}")

# ── 2) 폰트별 렌더링용 ttf 선택 (400에 가장 가까운 weight) ──
font_paths = {}
for path in tqdm(sorted(glob.glob(os.path.join(FONT_DIR, "*.ttf"))), desc="폰트 경로 매핑"):
    fname = os.path.splitext(os.path.basename(path))[0]
    parts = fname.rsplit("-", 1)
    if len(parts) != 2 or not parts[1].isdigit():
        continue
    name, weight = parts[0], int(parts[1])
    if name not in font_paths or abs(weight - 400) < abs(font_paths[name][1] - 400):
        font_paths[name] = (path, weight)

# ── 3) 클러스터별 이미지 생성 ──
for cid in tqdm(sorted(cluster_members.keys()), desc="클러스터 시각화"):
    members = cluster_members[cid]
    exemplar_name = font_names[exemplar_indices[cid]]

    # 대표 폰트를 맨 위로
    ordered = [exemplar_name] + [m for m in members if m != exemplar_name]

    img_height = PADDING + len(ordered) * ROW_HEIGHT + PADDING
    img = Image.new("RGB", (IMG_WIDTH, img_height), BG_COLOR)
    draw = ImageDraw.Draw(img)

    try:
        label_font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
    except Exception:
        label_font = ImageFont.load_default()

    for row_idx, fname in enumerate(ordered):
        y = PADDING + row_idx * ROW_HEIGHT
        is_exemplar = (fname == exemplar_name)

        # 대표 폰트 배경 강조
        if is_exemplar:
            draw.rectangle([0, y, IMG_WIDTH, y + ROW_HEIGHT], fill=EXEMPLAR_BG)

        # 폰트명 라벨
        weight_str = str(font_paths[fname][1]) if fname in font_paths else "?"
        label = f"{'★ ' if is_exemplar else ''}{fname} ({weight_str})"
        draw.text((PADDING, y + 20), label, fill=FG_COLOR, font=label_font)

        # 샘플 텍스트 렌더링
        if fname in font_paths:
            try:
                ttf = ImageFont.truetype(font_paths[fname][0], FONT_SIZE)
                draw.text((LABEL_WIDTH, y + 10), SAMPLE_TEXT, fill=FG_COLOR, font=ttf)
            except Exception:
                draw.text((LABEL_WIDTH, y + 20), "[렌더링 실패]", fill=(200, 0, 0), font=label_font)
        else:
            draw.text((LABEL_WIDTH, y + 20), "[폰트 없음]", fill=(200, 0, 0), font=label_font)

    save_path = os.path.join(OUTPUT_DIR, f"cluster_{cid:03d}_{len(ordered)}fonts.png")
    img.save(save_path)

print(f"\n저장 완료: {OUTPUT_DIR}/ ({n_clusters}개 이미지)")

클러스터 수: 79, 폰트 수: 641


폰트 경로 매핑:   0%|          | 0/1237 [00:00<?, ?it/s]

클러스터 시각화:   0%|          | 0/79 [00:00<?, ?it/s]


저장 완료: ./data/9_font_cluster_vis/ (79개 이미지)
